# MTP4Ge — Results Analysis

This notebook provides visualizations for:
1. Training / validation / test error rates (energy, force, stress)
2. Parity plots: MTP vs DFT
3. Active learning convergence
4. Pruning Pareto front (cost vs accuracy)
5. MD structural properties (RDF, MSD)

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

## 1. Error Summary

In [ ]:
def parse_errors(report_path: str) -> dict:
    """Parse energy/force/stress RMSE from mlp check_errors output."""
    text = Path(report_path).read_text()
    result = {}
    for line in text.splitlines():
        m = re.search(r'(energy|force|stress).*?([\d.eE+\-]+)\s*(meV|eV|GPa)', line, re.I)
        if m:
            result[m.group(1).lower()] = float(m.group(2))
    return result

splits = ['train', 'val', 'test']
errors = {}
for split in splits:
    report = f'results/errors/{split}/errors.txt'
    if Path(report).exists():
        errors[split] = parse_errors(report)
    else:
        print(f'No report for {split} — run scripts/02_test_errors.sh first')

if errors:
    print('\nError Summary:')
    print(f'{"Split":<8} {"Energy (meV/at)":>18} {"Force (eV/Å)":>14} {"Stress (GPa)":>14}')
    print('-' * 56)
    for split, e in errors.items():
        print(f'{split:<8} {e.get("energy", float("nan")):>18.3f} '
              f'{e.get("force", float("nan")):>14.4f} '
              f'{e.get("stress", float("nan")):>14.4f}')

## 2. Pruning Pareto Front

In [ ]:
import glob
import pandas as pd

pareto_files = sorted(glob.glob('results/pruning/pareto*/pareto_final_*.csv'))

if pareto_files:
    df = pd.read_csv(pareto_files[-1], header=None)
    print(f'Loaded Pareto front: {pareto_files[-1]}')
    print(f'  {len(df)} points on the Pareto front')
    print(df.head())

    fig, ax = plt.subplots(figsize=(7, 5))
    # Convention: col 0 = cost (FLOPs proxy), col 1 = loss (lower is better)
    ax.scatter(df.iloc[:, 0], df.iloc[:, 1], s=20, alpha=0.7)
    ax.set_xlabel('Cost (inference FLOPs proxy)')
    ax.set_ylabel('Validation loss')
    ax.set_title('Pruning Pareto Front')
    plt.tight_layout()
    plt.show()
else:
    print('No Pareto CSV found — run scripts/04_prune.sh first')

## 3. Parity Plots (MTP vs DFT)

In [ ]:
# TODO: parse efs_output.cfg from results/errors/test/efs_output.cfg
# and plot MTP-predicted energy vs DFT energy per atom.
print('Run: python src/test_errors.py --pot results/potentials/pot.almtp '
      '--cfg data/test.cfg --mode efs')